# 04 — Clean Window (Aug–Dec 2025)

Out-of-sample replication on the 2025 window.
Relevance stratification and agent vs lexicon comparison.

In [1]:
import sys, os
from pathlib import Path
for _cand in ['.', '..']:
    if (Path(_cand)/'src').is_dir() and (Path(_cand)/'legacy').is_dir():
        os.chdir(_cand); break
sys.path.insert(0, 'src'); sys.path.insert(0, 'legacy/src')

import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt

from src.stats_rigor import bootstrap_ic_ci, spearman_ic
from src.relevance_eval import export_label_sample
from src.grid import append_cells
from src.report_io import save_fig, save_table, setup_style
from src.config import PANELS_2025, AGENT_PANELS, COLORS

setup_style()
N_BOOT   = 1000
HORIZONS = [1, 5, 15]
print('Setup complete.')

Setup complete.


## 1. Load 2025 panels

In [2]:
frames_2025 = []
for ticker, path in PANELS_2025.items():
    if not Path(path).exists():
        print(f'  skip {ticker}: {path}')
        continue
    df = pd.read_csv(path)
    df['stock'] = ticker
    frames_2025.append(df)

panel_2025 = pd.concat(frames_2025, ignore_index=True) if frames_2025 else pd.DataFrame()
print(f'Clean-window panel: {len(panel_2025):,} rows')

if not frames_2025:
    print('WARNING: No 2025 panels found — clean-window section will be empty.')

Clean-window panel: 2,494 rows


## 2. Load agent panels

In [3]:
agent_frames = []
for key, path in AGENT_PANELS.items():
    if not Path(path).exists():
        print(f'  skip agent {key}: {path}')
        continue
    try:
        df = pd.read_parquet(path)
        df['_agent_key'] = key
        agent_frames.append(df)
    except Exception as e:
        print(f'  error loading {key}: {e}')
        continue

agent_panel = pd.concat(agent_frames, ignore_index=True) if agent_frames else pd.DataFrame()
print(f'Agent panel: {len(agent_panel):,} rows from {len(agent_frames)} keys')

# Report agent column presence
AGENT_COLS = ['agent_direction','agent_size','agent_horizon','agent_reasoning',
              'agent_signal','agent_relevance','agent_signal_agreement','agent_conviction_reasoning']
if not agent_panel.empty:
    present = [c for c in AGENT_COLS if c in agent_panel.columns]
    absent  = [c for c in AGENT_COLS if c not in agent_panel.columns]
    print(f'Agent cols present: {present}')
    if absent:
        print(f'  TODO(schema): missing agent cols: {absent}')

Agent panel: 3,998 rows from 5 keys
Agent cols present: ['agent_direction', 'agent_size', 'agent_horizon', 'agent_reasoning', 'agent_signal', 'agent_relevance', 'agent_signal_agreement', 'agent_conviction_reasoning']


## 3. Out-of-sample IC (clean window)

In [4]:
rows_oos = []
if not panel_2025.empty:
    panel_2025['ofi_x_llm'] = panel_2025['ofi_z'] * panel_2025['llm_score']
    for scorer_col, label in [('ofi_x_llm','OFI×LLM'), ('llm_score','LLM'), ('lm_score','LM')]:
        if scorer_col not in panel_2025.columns:
            continue
        for h in HORIZONS:
            ret_col = f'ret_{h}m'
            if ret_col not in panel_2025.columns:
                continue
            ci = bootstrap_ic_ci(panel_2025[scorer_col], panel_2025[ret_col], n_boot=N_BOOT, seed=42)
            rows_oos.append({'scorer': label, 'horizon': h, 'window': '2025', **ci})

tbl_oos = pd.DataFrame(rows_oos)
if not tbl_oos.empty:
    display_cols = ['scorer','horizon','ic','ci_lo','ci_hi','p_boot','n']
    print(tbl_oos[display_cols].to_string(index=False, float_format='{:.4f}'.format))
    save_table(tbl_oos[display_cols], '04_oos_ic',
               caption='Out-of-sample IC (Aug-Dec 2025 window). Bootstrap CI n\_boot=1000.',
               label='tab:oos_ic')

 scorer  horizon      ic   ci_lo  ci_hi  p_boot    n
OFI×LLM        1  0.0251 -0.0161 0.0663  0.1160 2480
OFI×LLM        5  0.0202 -0.0181 0.0613  0.1500 2462
OFI×LLM       15  0.0112 -0.0301 0.0504  0.2990 2423
    LLM        1  0.0031 -0.0365 0.0463  0.4450 2480
    LLM        5  0.0337 -0.0066 0.0739  0.0520 2462
    LLM       15  0.0059 -0.0325 0.0500  0.3590 2423
     LM        1 -0.0099 -0.0492 0.0307  0.6610 2480
     LM        5 -0.0214 -0.0616 0.0161  0.8660 2462
     LM       15  0.0319 -0.0073 0.0699  0.0610 2423
  Saved table  → results/tables/04_oos_ic.csv + results/tables/04_oos_ic.tex


## 4. Relevance stratification on agent panel

In [5]:
rows_rel = []
if not agent_panel.empty and 'agent_relevance' in agent_panel.columns:
    agent_panel['ofi_x_llm'] = agent_panel['ofi_z'] * agent_panel['llm_score']
    buckets = agent_panel['agent_relevance'].unique()
    print(f'Relevance buckets: {sorted(buckets)}')

    for bucket in sorted(buckets):
        sub = agent_panel[agent_panel['agent_relevance'] == bucket]
        for h in HORIZONS:
            ret_col = f'ret_{h}m'
            if ret_col not in sub.columns or len(sub) < 10:
                continue
            ci = bootstrap_ic_ci(sub['ofi_x_llm'], sub[ret_col], n_boot=N_BOOT, seed=42)
            rows_rel.append({'relevance': bucket, 'horizon': h, **ci})

    if rows_rel:
        tbl_rel = pd.DataFrame(rows_rel)
        display_cols = ['relevance','horizon','ic','ci_lo','ci_hi','p_boot','n']
        print(tbl_rel[display_cols].to_string(index=False, float_format='{:.4f}'.format))
        save_table(tbl_rel[display_cols], '04_relevance_ic',
                   caption='IC stratified by agent relevance bucket (ofi\_x\_llm).',
                   label='tab:relevance_ic')
else:
    print('  Skipping relevance stratification: agent panel empty or missing agent_relevance column.')
    print('  TODO(schema): confirm agent_relevance column name')

Relevance buckets: ['direct', 'macro', 'sector', 'unrelated']


relevance  horizon      ic   ci_lo  ci_hi  p_boot    n
   direct        1 -0.0067 -0.0656 0.0537  0.6040 1085
   direct        5 -0.0025 -0.0588 0.0636  0.5120 1076
   direct       15  0.0061 -0.0535 0.0719  0.4110 1057
    macro        1 -0.0188 -0.0923 0.0517  0.7130  773
    macro        5  0.0029 -0.0747 0.0702  0.4880  768
    macro       15  0.0320 -0.0439 0.1019  0.2030  754
   sector        1  0.0372 -0.0305 0.1068  0.1270  862
   sector        5  0.0567 -0.0068 0.1263  0.0400  857
   sector       15  0.0535 -0.0163 0.1179  0.0650  843
unrelated        1  0.0460 -0.0081 0.1024  0.0510 1271
unrelated        5 -0.0157 -0.0728 0.0377  0.6960 1259
unrelated       15 -0.0108 -0.0671 0.0462  0.6280 1231
  Saved table  → results/tables/04_relevance_ic.csv + results/tables/04_relevance_ic.tex


## 5. Agent vs lexicon comparison

In [6]:
rows_cmp = []
for ticker, path in PANELS_2025.items():
    if not Path(path).exists():
        continue
    df = pd.read_csv(path)
    df['stock'] = ticker
    df['ofi_x_llm'] = df['ofi_z'] * df['llm_score']
    df['ofi_x_lm']  = df['ofi_z'] * df['lm_score']
    for scorer_col, label in [('ofi_x_llm','OFI×LLM'), ('ofi_x_lm','OFI×LM'), ('llm_score','LLM'), ('lm_score','LM')]:
        if scorer_col not in df.columns:
            continue
        for h in HORIZONS:
            ret_col = f'ret_{h}m'
            if ret_col not in df.columns:
                continue
            ci = bootstrap_ic_ci(df[scorer_col], df[ret_col], n_boot=N_BOOT, seed=42)
            rows_cmp.append({'ticker': ticker, 'scorer': label, 'horizon': h, **ci})

if rows_cmp:
    tbl_cmp = pd.DataFrame(rows_cmp)
    display_cols = ['ticker','scorer','horizon','ic','ci_lo','ci_hi','p_boot','n']
    print(tbl_cmp[display_cols].to_string(index=False, float_format='{:.4f}'.format))

    # Flag where LM beats LLM
    pivot = tbl_cmp.pivot_table(index=['ticker','horizon'], columns='scorer', values='ic')
    if 'OFI×LM' in pivot.columns and 'OFI×LLM' in pivot.columns:
        beats = pivot[pivot['OFI×LM'] > pivot['OFI×LLM']]
        if not beats.empty:
            print('\n  Cases where LM beats LLM:')
            print(beats[['OFI×LM','OFI×LLM']].to_string(float_format='{:.4f}'.format))

    save_table(tbl_cmp[display_cols], '04_agent_vs_lexicon',
               caption='Per-ticker IC comparison: LLM vs LM lexicon (2025 window). Cases where lexicon wins are noted.',
               label='tab:agent_vs_lexicon')

ticker  scorer  horizon      ic   ci_lo   ci_hi  p_boot   n
  AAPL OFI×LLM        1  0.0451 -0.0495  0.1335  0.1670 507
  AAPL OFI×LLM        5  0.0198 -0.0771  0.1082  0.3510 502
  AAPL OFI×LLM       15  0.0136 -0.0780  0.0971  0.3880 498
  AAPL  OFI×LM        1 -0.0024 -0.0873  0.0846  0.5010 507
  AAPL  OFI×LM        5  0.0008 -0.0883  0.0970  0.4700 502
  AAPL  OFI×LM       15 -0.0675 -0.1584  0.0320  0.9200 498
  AAPL     LLM        1  0.0152 -0.0756  0.1047  0.3610 507
  AAPL     LLM        5 -0.0240 -0.1132  0.0689  0.7080 502
  AAPL     LLM       15 -0.0786 -0.1620  0.0142  0.9480 498
  AAPL      LM        1 -0.0426 -0.1305  0.0416  0.8260 507
  AAPL      LM        5 -0.0155 -0.1015  0.0746  0.6390 502
  AAPL      LM       15 -0.0224 -0.1102  0.0633  0.6770 498
   AMD OFI×LLM        1  0.0131 -0.1120  0.1461  0.4040 271
   AMD OFI×LLM        5  0.0134 -0.1184  0.1448  0.4290 271
   AMD OFI×LLM       15 -0.0071 -0.1333  0.1132  0.5710 264
   AMD  OFI×LM        1  0.0226 -0.0956 

## 6. Export relevance label sample

In [7]:
if not agent_panel.empty:
    export_label_sample(agent_panel, k=50)

  Saved 48-event label sample → results/tables/relevance_sample_to_label.csv
  Bucket counts: {'macro': 12, 'direct': 12, 'unrelated': 12, 'sector': 12}


## 7. Relevance IC figure

In [8]:
if rows_rel:
    fig, ax = plt.subplots(figsize=(7, 4))
    buckets_found = sorted(tbl_rel['relevance'].unique())
    x = np.arange(len(HORIZONS))
    width = 0.8 / max(len(buckets_found), 1)
    for i, bucket in enumerate(buckets_found):
        sub = tbl_rel[tbl_rel['relevance'] == bucket].sort_values('horizon')
        ics = sub['ic'].values
        los = ics - sub['ci_lo'].values
        his = sub['ci_hi'].values - ics
        offset = (i - len(buckets_found)/2 + 0.5) * width
        ax.bar(x + offset, ics, width=width*0.9, label=bucket,
               yerr=[los, his], error_kw={'linewidth': 0.7, 'capsize': 3})
    ax.set_xticks(x)
    ax.set_xticklabels([f'{h}-min' for h in HORIZONS])
    ax.axhline(0, color='black', lw=0.7, ls='--', alpha=0.5)
    ax.set_ylabel('Spearman IC')
    ax.set_title('IC by relevance bucket (ofi×llm, agent panel)')
    ax.legend(frameon=False)
    ax.grid(axis='y', alpha=0.2)
    save_fig(fig, '04_relevance_ic')
    plt.close()

  Saved figure → results/figures/04_relevance_ic.pdf


## 8. Append to BH grid

In [9]:
grid_rows = []
for _, row in tbl_oos.iterrows() if rows_oos else [].__iter__():
    grid_rows.append({
        'notebook': '04', 'cell_id': f"oos_{row['scorer']}_{row['horizon']}m",
        'stock': 'pooled_2025', 'scorer': row['scorer'], 'horizon': row['horizon'],
        'n': row['n'], 'ic': row['ic'], 'ci_lo': row['ci_lo'], 'ci_hi': row['ci_hi'],
        'p': row['p_boot'],
    })
for _, row in (tbl_rel.iterrows() if rows_rel else [].__iter__()):
    grid_rows.append({
        'notebook': '04', 'cell_id': f"rel_{row['relevance']}_{row['horizon']}m",
        'stock': 'agent_pooled', 'scorer': f"ofi_x_llm|{row['relevance']}",
        'horizon': row['horizon'], 'n': row['n'], 'ic': row['ic'],
        'ci_lo': row['ci_lo'], 'ci_hi': row['ci_hi'], 'p': row['p_boot'],
    })
if rows_cmp:
    for _, row in tbl_cmp.iterrows():
        grid_rows.append({
            'notebook': '04', 'cell_id': f"cmp_{row['ticker']}_{row['scorer']}_{row['horizon']}m",
            'stock': row['ticker'], 'scorer': row['scorer'], 'horizon': row['horizon'],
            'n': row['n'], 'ic': row['ic'], 'ci_lo': row['ci_lo'], 'ci_hi': row['ci_hi'],
            'p': row['p_boot'],
        })
if grid_rows:
    append_cells(grid_rows)
    print(f'Appended {len(grid_rows)} cells to secondary grid.')

Appended 81 cells to secondary grid.
